In [7]:
import sys
sys.path.append("/Users/bubble/Desktop/Project/Buckling/non_hookean/Layout")
import math
import gdsfactory as gf
from gdsfactory.generic_tech import get_generic_pdk
PDK = get_generic_pdk()
PDK.activate()
cell_temp = gf.Component()

In [8]:
'''
layer 1: thick ald layer
layer 2: thin ald layer
layer 3: KOH layer
layer 4: KOH (backside asymmetric) layer
'''

'\nlayer 1: thick ald layer\nlayer 2: thin ald layer\nlayer 3: KOH layer\nlayer 4: KOH (backside asymmetric) layer\n'

In [9]:
# get the horizontal distance caused by KOH etching.
def KOHSize(height):
    size = height / math.tan(math.radians(54.74))
    return round(size, 2)
# print(KOHSize(525)*2)
block = gf.Component()

# thick ald layer pattern
length = 14000
width = 500
rec = gf.components.rectangle(size=(length, width), layer=1)
n = 2
gap = 9000
for i in range(n):
    (block << rec).move(
        ((15000-length) / 2, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 )) 
    
# block.show()

In [ ]:
# thin ald layer pattern
# frame of pattern for boolean operation
length_2 = 14010
width_2 = 502
rec = gf.components.rectangle(size=(length_2, width_2), layer=(10, 0))
gap = 9000
for i in range(n):
    (block << rec).move(
        ((15000-length_2) / 2, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(width_2-width)/2)) 
    
k = 9 # number beam group  
marker_array = []
for i in range(k):
    marker_height = 5
    width_temp = 10
    marker_single = gf.Component()
    single_rec = gf.components.rectangle(size=(width_temp, marker_height), layer=(20, 0))
    for j in range(i+1):
        (marker_single << single_rec).move(((j+1)*200/(k+1), 0))
    marker_array.append(marker_single)

# beam pattern
w_list = [5, 10, 15, 20, 200]
L = 20
gap_horizental = 250
def get_w_list(n_group):
    w_group = gf.Component()
    for i in range(len(w_list)):
        rec = gf.components.rectangle(size=(w_list[i], L), layer=(2, 0))
        if i != len(w_list)-1:
            (w_group << rec).movex((i+1)*gap_horizental - w_list[i]/2)
        else:
            (w_group << rec).movex(i*gap_horizental + 300)
            (w_group << marker_array[n_group]).move((i*gap_horizental + 300, 20-marker_height))
            # marker = gf.components.text(text=f"G {n_group+1}", size=50, layer=(1, 0)) # on thicker ald layer
            # (w_group << marker).move((i*gap_horizental + 300, -100))
    return w_group

for i in range(n):
    for j in range(k):
        (block << get_w_list(j)).move((length / k * j + (15000-length) / 2, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(width_2-width)/2))
        (block << get_w_list(j)).move((length / k * j + (15000-length) / 2, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(width_2-width)/2)).dmirror_y(i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 + width/2)
       


# block.show()

In [11]:
# backside etching pattern
length_etch = length + KOHSize(525)*2
width_etch = width + KOHSize(525)*2
rec_etch = gf.components.rectangle(size=(length_etch, width_etch), layer=(3, 0))
rec_clip = gf.components.rectangle(size=(1500, 375), layer=(3, 0))

for i in range(n):
    # hole
    (block << rec_etch).move(
        ((15000-length_etch) / 2, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(width_etch-width)/2))
    # clip
    (block << rec_clip).move(
        (-2000, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(375-width)/2))
    (block << rec_clip).move(
        (15500, i*(width+gap)+(15000-(n-1)*gap-n*width) / 2 -(375-width)/2))
# block.show()

In [ ]:
# boolean operation
# Thicker ALD
thick = gf.boolean(A = block, B = block,  operation="not", layer1=(1, 0), layer2=(15, 0), layer=(1, 0))
cell_temp << thick

# Thinner ALD
temp = gf.boolean(A = block, B = block,  operation="not", layer1=(2, 0), layer2=(20, 0), layer=(2, 0))
thinn = gf.boolean(A = block, B = temp,  operation="not", layer1=(10, 0), layer2=(2, 0), layer=(2, 0))
cell_temp << thinn

# Backside etching
backside = gf.boolean(A = block, B = block,  operation="not", layer1=(3, 0), layer2=(15, 0), layer=(3, 0))
cell_temp << backside


# frame
frame1 = gf.components.rectangle(size=(15000, 15000), layer=(20, 0))
frame2 = gf.components.rectangle(size=(20000, 20000), layer=(21, 0))
frame2_ref = cell_temp << frame2
frame2_ref.move((-2500, -2500))
cell_temp << frame1

cell_non_hookean = gf.Component()
cell_ref = cell_non_hookean << cell_temp
cell_ref.move((2500, -7500))
# cell_non_hookean.show()
# cell_non_hookean.write_gds("mesh.gds")
# cell_non_hookean.plot()